In [16]:
import pandas as pd
import sqlite3
import glob
from pathlib import Path

In [17]:
# Find all CSV files from day folders
csv_files = sorted(glob.glob("data/amazon/day_*/amazon_*.csv"))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f}")
 
# Read and merge
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
 
print(f"\nMerged all CSVs")
print(f"  Total rows: {len(df)}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Date range: {df['scrape_day'].min()} to {df['scrape_day'].max()}")

Found 18 CSV files:
  data/amazon\day_2026-04-26\amazon_2026-04-26.csv
  data/amazon\day_2026-04-27\amazon_2026-04-27.csv
  data/amazon\day_2026-04-28\amazon_2026-04-28.csv
  data/amazon\day_2026-04-29\amazon_2026-04-29.csv
  data/amazon\day_2026-04-30\amazon_2026-04-30.csv
  data/amazon\day_2026-05-11\amazon_2026-05-11.csv
  data/amazon\day_2026-05-13\amazon_2026-05-13.csv
  data/amazon\day_2026-05-14\amazon_2026-05-14.csv
  data/amazon\day_2026-05-15\amazon_2026-05-15.csv
  data/amazon\day_2026-05-16\amazon_2026-05-16.csv
  data/amazon\day_2026-05-17\amazon_2026-05-17.csv
  data/amazon\day_2026-05-18\amazon_2026-05-18.csv
  data/amazon\day_2026-05-19\amazon_2026-05-19.csv
  data/amazon\day_2026-05-20\amazon_2026-05-20.csv
  data/amazon\day_2026-05-21\amazon_2026-05-21.csv
  data/amazon\day_2026-05-22\amazon_2026-05-22.csv
  data/amazon\day_2026-05-23\amazon_2026-05-23.csv
  data/amazon\day_2026-05-24\amazon_2026-05-24.csv

Merged all CSVs
  Total rows: 13478
  Total columns: 14
  Dat

In [18]:
print("Column names:")
print(df.columns.tolist())
 
print("\nFirst 3 rows:")
df.head(3)

Column names:
['name', 'brand', 'category', 'price', 'mrp', 'discount_pct', 'rating', 'reviews', 'in_stock', 'delivery_info', 'seller_count', 'source', 'scrape_day', 'timestamp']

First 3 rows:


,name,brand,category,price,mrp,discount_pct,rating,reviews,in_stock,delivery_info,seller_count,source,scrape_day,timestamp
0,"Lava Bold N2 (Siachen White, 4 GB RAM, 64 GB S...",Lava,smartphones,8899.0,9499.0,6.0,3.6,59.0,1,"FREE delivery Thu, 30 Apr",NaN,amazon,2026-04-26,2026-04-26T23:00:49.123130
1,"Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...",Redmi,smartphones,11999.0,11999.0,NaN,4.0,5723.0,1,FREE delivery 28 Apr - 7 May,NaN,amazon,2026-04-26,2026-04-26T23:00:49.799410
2,"realme C71 4G Smartphone 6GB+128GB,6.745 inch ...",realme,smartphones,12199.0,12199.0,NaN,4.5,297.0,1,"FREE delivery Sat, 2 May",NaN,amazon,2026-04-26,2026-04-26T23:00:50.060998


In [19]:
# Convert to correct data types
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['mrp'] = pd.to_numeric(df['mrp'], errors='coerce')
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')
df['reviews'] = pd.to_numeric(df['reviews'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['scrape_day'] = pd.to_datetime(df['scrape_day'])
 
# Check for nulls in critical columns
print("Nulls in critical columns:")
print(f"  price: {df['price'].isna().sum()}")
print(f"  mrp: {df['mrp'].isna().sum()}")
print(f"  name: {df['name'].isna().sum()}")
print(f"  brand: {df['brand'].isna().sum()}")
 
# Drop rows where price or mrp is null
df = df.dropna(subset=['price', 'mrp', 'name', 'brand'])
 
print(f"\n✓ After dropping nulls: {len(df)} rows")
 
# Recalculate missing discount_pct
missing_discount = df['discount_pct'].isna().sum()
df['discount_pct'] = df['discount_pct'].fillna(
    round((df['mrp'] - df['price']) / df['mrp'] * 100, 1)
)
print(f" Filled {missing_discount} missing discount_pct values")

Nulls in critical columns:
  price: 601
  mrp: 601
  name: 0
  brand: 0

✓ After dropping nulls: 12877 rows
 Filled 912 missing discount_pct values


In [20]:
print("Data Types:")
print(df.dtypes)
 
print("\nShape:", df.shape)
 
print("\nUnique values:")
print(f"  Unique products (by name): {df['name'].nunique()}")
print(f"  Unique brands: {df['brand'].nunique()}")
print(f"  Categories: {df['category'].unique()}")
print(f"  Days scraped: {df['scrape_day'].nunique()}")
 

Data Types:
name                     object
brand                    object
category                 object
price                   float64
mrp                     float64
discount_pct            float64
rating                  float64
reviews                 float64
in_stock                  int64
delivery_info            object
seller_count            float64
source                   object
scrape_day       datetime64[ns]
timestamp                object
dtype: object

Shape: (12877, 14)

Unique values:
  Unique products (by name): 2126
  Unique brands: 427
  Categories: ['smartphones' 'laptops' 'home_appliances']
  Days scraped: 18


In [21]:
df.head(3)

,name,brand,category,price,mrp,discount_pct,rating,reviews,in_stock,delivery_info,seller_count,source,scrape_day,timestamp
0,"Lava Bold N2 (Siachen White, 4 GB RAM, 64 GB S...",Lava,smartphones,8899.0,9499.0,6.0,3.6,59.0,1,"FREE delivery Thu, 30 Apr",NaN,amazon,2026-04-26,2026-04-26T23:00:49.123130
1,"Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...",Redmi,smartphones,11999.0,11999.0,0.0,4.0,5723.0,1,FREE delivery 28 Apr - 7 May,NaN,amazon,2026-04-26,2026-04-26T23:00:49.799410
2,"realme C71 4G Smartphone 6GB+128GB,6.745 inch ...",realme,smartphones,12199.0,12199.0,0.0,4.5,297.0,1,"FREE delivery Sat, 2 May",NaN,amazon,2026-04-26,2026-04-26T23:00:50.060998


In [22]:
# Create database file
db_path = "amazon_analysis.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
 
print(f"Created database: {db_path}")
 
# ---- TABLE 1: PRODUCTS (static info) ----
# One row per unique product
products = df[['name', 'brand', 'category']].drop_duplicates(subset=['name']).reset_index(drop=True)
products['product_id'] = range(1, len(products) + 1)
 
# Reorder columns
products = products[['product_id', 'name', 'brand', 'category']]
 
# Save to database
products.to_sql('products', conn, if_exists='replace', index=False)
print(f"✓ Created 'products' table: {len(products)} products")
 
# ---- TABLE 2: PRICE_HISTORY (daily snapshots) ----
# Merge product_id into main dataframe
df_with_id = df.merge(products[['product_id', 'name']], on='name', how='left')
 
# Select relevant columns for price history
price_history = df_with_id[['product_id', 'scrape_day', 'price', 'mrp', 'discount_pct']].copy()
price_history = price_history.rename(columns={'scrape_day': 'date'})
 
# Save to database
price_history.to_sql('price_history', conn, if_exists='replace', index=False)
print(f"✓ Created 'price_history' table: {len(price_history)} price snapshots")
 
# ---- TABLE 3: MARKET_SIGNALS (demand & supply signals) ----
# Select relevant columns for signals
market_signals = df_with_id[['product_id', 'scrape_day', 'reviews', 'rating', 'in_stock']].copy()
market_signals = market_signals.rename(columns={'scrape_day': 'date'})
 
# Save to database
market_signals.to_sql('market_signals', conn, if_exists='replace', index=False)
print(f"✓ Created 'market_signals' table: {len(market_signals)} signal records")
 
conn.commit()
print("\n✓ Database saved successfully!")

Created database: amazon_analysis.db
✓ Created 'products' table: 2126 products
✓ Created 'price_history' table: 12877 price snapshots
✓ Created 'market_signals' table: 12877 signal records

✓ Database saved successfully!


In [23]:
# Check what tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"Tables in database: {[t[0] for t in tables]}")
 
# Show table schemas
print("\n--- PRODUCTS TABLE ---")
cursor.execute("PRAGMA table_info(products);")
for row in cursor.fetchall():
    print(f"  {row[1]}: {row[2]}")
 
print("\n--- PRICE_HISTORY TABLE ---")
cursor.execute("PRAGMA table_info(price_history);")
for row in cursor.fetchall():
    print(f"  {row[1]}: {row[2]}")
 
print("\n--- MARKET_SIGNALS TABLE ---")
cursor.execute("PRAGMA table_info(market_signals);")
for row in cursor.fetchall():
    print(f"  {row[1]}: {row[2]}")

Tables in database: ['products', 'price_history', 'market_signals']

--- PRODUCTS TABLE ---
  product_id: INTEGER
  name: TEXT
  brand: TEXT
  category: TEXT

--- PRICE_HISTORY TABLE ---
  product_id: INTEGER
  date: TIMESTAMP
  price: REAL
  mrp: REAL
  discount_pct: REAL

--- MARKET_SIGNALS TABLE ---
  product_id: INTEGER
  date: TIMESTAMP
  reviews: REAL
  rating: REAL
  in_stock: INTEGER


In [24]:
print("\n" + "="*60)
print("SANITY CHECKS")
print("="*60)
 
# Query 1: Total products and brands
result = pd.read_sql("SELECT COUNT(*) as total_products FROM products", conn)
print(f"\n1. Total unique products: {result['total_products'].values[0]}")
 
result = pd.read_sql("SELECT COUNT(DISTINCT brand) as total_brands FROM products", conn)
print(f"2. Total unique brands: {result['total_brands'].values[0]}")
 
# Query 2: Products per category
result = pd.read_sql("SELECT category, COUNT(*) as count FROM products GROUP BY category", conn)
print(f"\n3. Products per category:")
print(result.to_string(index=False))
 
# Query 3: Avg discount per category
result = pd.read_sql(
    "SELECT p.category, ROUND(AVG(ph.discount_pct), 1) as avg_discount "
    "FROM price_history ph "
    "JOIN products p ON ph.product_id = p.product_id "
    "GROUP BY p.category",
    conn
)
print(f"\n4. Average discount by category:")
print(result.to_string(index=False))
 
# Query 4: Top 10 brands by product count
result = pd.read_sql(
    "SELECT brand, COUNT(*) as count FROM products GROUP BY brand ORDER BY count DESC LIMIT 10",
    conn
)
print(f"\n5. Top 10 brands:")
print(result.to_string(index=False))
 
conn.close()


SANITY CHECKS

1. Total unique products: 2126
2. Total unique brands: 427

3. Products per category:
       category  count
home_appliances    857
        laptops    588
    smartphones    681

4. Average discount by category:
       category  avg_discount
home_appliances          46.1
        laptops          21.0
    smartphones          15.1

5. Top 10 brands:
   brand  count
      HP    182
  Lenovo    128
 Samsung    120
    ASUS     95
  realme     82
Motorola     74
Onsitego     71
    vivo     47
    acer     45
   REDMI     39
